# 로짓 덤프 — 3모델 × {test, val}

- 산출물: `logits_{tag}_{split}.npy`(`[N, 188]` fp32) + `doc_ids_{split}.json`
- 로짓 행 순서 = `doc_ids_{split}.json` 순서 = 토큰화 데이터셋 행 순서

In [ ]:
!wget "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [2]:
import os
import gc
import json
import random
from pathlib import Path

import numpy as np
from numpy.typing import NDArray
from dotenv import load_dotenv

load_dotenv()

# 로컬
# ROOT = Path(os.environ["DATA_ROOT"])
# os.environ["HF_HOME"] = str(ROOT / ".hf_cache")

# 코랩
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
os.environ["HF_HOME"] = "/content/.hf_cache"

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Config
config = {
    "seed": 42,
    "num_labels": 188,
    "splits": ["test", "val"],
    "fields": ["invention_title", "ipc_main", "abstract", "claims"],    # 기록용
    "hf_cache": "/content/.hf_cache",                                   # 데이터셋 캐시는 로컬 디스크(Drive I/O는 느리다)
    "out_path": "/content/drive/MyDrive/patent_disc/output/",           # 로짓은 Drive 직결 = 휘발 방지 + resume
    "ssot_path": "/content/drive/MyDrive/patent_disc/",                 # total_metrics_{tag}.json 위치(검증용)
}

# 모델별 사양
#
# ⚠️ batch_size는 8 고정 — 평가 노트북(03_02·04_03·05_02)이 전부 8이다. 동적 패딩(padding=True)은
#    배치 내 최장 문서에 맞추므로 배치 크기가 바뀌면 패딩량과 행렬 shape이 바뀌고, fp16·bf16 autocast의
#    누산 순서·커널 선택이 달라져 로짓이 ~1e-4 흔들린다. τ=0.5 경계와 top-1 argmax에서 문서 몇 건이
#    뒤집혀 지표 4번째 자리가 어긋난다(실측: 64로 올렸더니 mb512 anchor F1 0.8203→0.8199).
MODELS = [
    {
        "tag": "kobert-patent-baseline_len512",
        "arch": "kobert",
        "ckpt": "ingyoun/kobert-patent-baseline",
        "tokenizer": "monologg/kobert",
        "tok_rev": "38279184ba645e8c94d709fbe92eb5bcb47312c1",
        "token_ds": "ingyoun/patent-clean-text-kobert-tokenized",
        "max_len": None,           # 전처리(02_01)에서 이미 truncation=True, max_length=512로 잘려 올라갔다
        "batch_size": 8,
    },
    {
        "tag": "modernbert-patent-len512",
        "arch": "modernbert",
        "ckpt": "ingyoun/A.X-patent-maxlen512",
        "tokenizer": "skt/A.X-Encoder-base",
        "tok_rev": "9708f9c404ace91efd25c06fac2d73413616f4ef",
        "token_ds": "ingyoun/patent-clean-text-modernbert-tokenized",
        "max_len": 512,            # 토큰화 데이터셋은 절단 없이 올라가 있다 — 훈련 창으로 맞춘다
        "batch_size": 8,
    },
    {
        "tag": "modernbert-patent-len8192",
        "arch": "modernbert",
        "ckpt": "ingyoun/A.X-patent-maxlen8192",
        "tokenizer": "skt/A.X-Encoder-base",
        "tok_rev": "9708f9c404ace91efd25c06fac2d73413616f4ef",
        "token_ds": "ingyoun/patent-clean-text-modernbert-tokenized",
        "max_len": 8192,
        "batch_size": 8,
    },
]

In [5]:
# fix seed
random.seed(config["seed"])
np.random.seed(config["seed"])
torch.manual_seed(config["seed"])
torch.cuda.manual_seed_all(config["seed"])

In [6]:
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

NVIDIA L4


## 추론

In [7]:
class EvalCollator:
    """동적 패딩"""
    def __init__(self, tokenizer):
        self.tok = tokenizer

    def __call__(self, feats):
        enc = [
            {"input_ids": f["input_ids"],
            "attention_mask": f["attention_mask"]}
            for f in feats
        ]
        return self.tok.pad(enc, padding=True, return_tensors="pt")


class LogitsRunner:
    """모델 1개 → split별 logits를 추론·캐시"""
    def __init__(self, model, tokenizer, cache_dir, tag: str, arch: str, batch_size: int):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = model.to(self.device).eval()
        self.collate = EvalCollator(tokenizer)
        self.cache_dir = Path(cache_dir)
        self.tag, self.arch, self.batch_size = tag, arch, batch_size

    @torch.no_grad
    def _infer(self, ds) -> NDArray:
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False, collate_fn=self.collate)
        dtype = torch.bfloat16 if self.arch == "modernbert" else torch.float16
        chunks = []
        for enc in tqdm(loader, desc=self.tag):
            enc = {k: v.to(self.device) for k, v in enc.items()}
            if self.device == "cuda":
                with torch.autocast("cuda", dtype=dtype):
                    logits = self.model(**enc).logits
            else:
                logits = self.model(**enc).logits
            chunks.append(logits.float().cpu().numpy())
        return np.concatenate(chunks, axis=0)

    def get(self, ds, split) -> NDArray:
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        fp = self.cache_dir / f"logits_{self.tag}_{split}.npy"
        if fp.exists():
            print(f"[skip] {fp.name}")
            return np.load(fp)
        arr = self._infer(ds)
        np.save(fp, arr)
        print(f"[save] {fp.name} {arr.shape}")
        return arr

## Orchestrator

In [8]:
def save_doc_ids(out_path, split, doc_ids):
    """split별 document_id 축을 1회 저장하고, 이후 모델은 같은 순서인지 검증한다.
    로짓 행 순서 = 세 모델의 토큰화 데이터셋은 같은 clean-text에서 같은 순서로 파생
    """
    fp = Path(out_path) / f"doc_ids_{split}.json"
    fp.parent.mkdir(parents=True, exist_ok=True)
    if fp.exists():
        assert json.loads(fp.read_text()) == list(doc_ids), f"{split}: document_id 순서가 모델 간 불일치"
        return
    fp.write_text(json.dumps(list(doc_ids)))


class LogitDumpHarness:
    """모델 사양 1개 → 지정 split들의 로짓 덤프."""
    def __init__(self, spec: dict, config: dict):
        self.spec, self.cfg = spec, config

    @staticmethod
    def _truncate(ds, tokenizer, max_len):
        """ max_len으로 절단 — 선두 <s> 유지 + 꼬리를 eos로 마감"""
        eos_id = tokenizer.eos_token_id
        def _fn(batch):
            ids, masks = [], []
            for x, m in zip(batch["input_ids"], batch["attention_mask"]):
                if len(x) > max_len:
                    x = x[: max_len - 1] + [eos_id]
                    m = m[:max_len]
                ids.append(x)
                masks.append(m)
            return {"input_ids": ids, "attention_mask": masks}
        return ds.map(_fn, batched=True)

    def _load_tokenizer(self):
        kw = {"revision": self.spec["tok_rev"]}
        if self.spec["arch"] == "kobert":
            kw["trust_remote_code"] = True
        return AutoTokenizer.from_pretrained(self.spec["tokenizer"], **kw)

    def _load_model(self):
        if self.spec["arch"] == "modernbert":
            return AutoModelForSequenceClassification.from_pretrained(
                pretrained_model_name_or_path=self.spec["ckpt"],
                dtype=torch.float32,                   # fp32 로드 + autocast bf16 (평가 프로토콜)
                attn_implementation="flash_attention_2",
            )
        return AutoModelForSequenceClassification.from_pretrained(
            pretrained_model_name_or_path=self.spec["ckpt"]
        )

    def run(self, splits):
        cfg, spec = self.cfg, self.spec
        tokenizer = self._load_tokenizer()
        model = self._load_model()
        runner = LogitsRunner(
            model=model, tokenizer=tokenizer, cache_dir=cfg["out_path"],
            tag=spec["tag"], arch=spec["arch"], batch_size=spec["batch_size"],
        )
        for split in splits:
            ds = load_dataset(spec["token_ds"], cache_dir=cfg["hf_cache"], split=split)
            save_doc_ids(cfg["out_path"], split, ds["document_id"])
            n = len(ds)
            if spec["max_len"] is not None:            # KoBERT 데이터셋은 이미 512로 잘려 있다
                ds = self._truncate(ds, tokenizer, spec["max_len"])
            logits = runner.get(ds, split)
            assert logits.shape == (n, cfg["num_labels"]), f"{spec['tag']}/{split}: {logits.shape} != ({n}, {cfg['num_labels']})"

        del runner, model
        gc.collect()
        torch.cuda.empty_cache()

## 실행

In [ ]:
for spec in MODELS:
    print(f"\n=== {spec['tag']} ===")
    LogitDumpHarness(spec, config).run(config["splits"])

## 검증

test 로짓으로 headline 지표를 재계산 `total_metrics_{tag}.json`과 4자리 일치를 확인

In [10]:
def verify_test(spec, config) -> bool:
    logits = np.load(Path(config["out_path"]) / f"logits_{spec['tag']}_test.npy")
    ds = load_dataset(spec["token_ds"], cache_dir=config["hf_cache"], split="test")
    Y = np.asarray(ds["labels"], dtype=int)
    pred = (1.0 / (1.0 + np.exp(-logits)) >= 0.5).astype(int)
    got = {
        "micro": f1_score(Y, pred, average="micro", zero_division=0),
        "macro": f1_score(Y, pred, average="macro", zero_division=0),
        "sample": f1_score(Y, pred, average="samples", zero_division=0),
        "empty_rate": float((pred.sum(1) == 0).mean()),
        "anchor_weighted_f1": f1_score(Y.argmax(1), logits.argmax(1), average="weighted", zero_division=0),
    }

    fp = Path(config["ssot_path"]) / f"total_metrics_{spec['tag']}.json"
    if not fp.exists():
        print(f"  [warn] SSOT 없음: {fp} — 로컬 output/과 대조할 것")
        print("  " + "  ".join(f"{k}={v:.4f}" for k, v in got.items()))
        return False

    ref = json.loads(fp.read_text(encoding="utf-8"))
    want = {
        **ref["multilabel_f1"]["keep"],
        "empty_rate": ref["empty_rate_tau_micro"],
        "anchor_weighted_f1": ref["anchor_top1"]["weighted_f1"],
    }
    ok = True
    for k, v in got.items():
        hit = round(v, 4) == round(want[k], 4)
        ok &= hit
        print(f"  {'OK  ' if hit else 'FAIL'} {k:20s} got={v:.4f} ssot={want[k]:.4f}")
    return ok


for spec in MODELS:
    print(f"\n=== {spec['tag']} ===")
    verify_test(spec, config)


=== kobert-patent-baseline_len512 ===
  [warn] SSOT 없음: /content/drive/MyDrive/patent_disc/total_metrics_kobert-patent-baseline_len512.json — 로컬 output/과 대조할 것
  micro=0.8502  macro=0.8470  sample=0.8656  empty_rate=0.0116  anchor_weighted_f1=0.8148

=== modernbert-patent-len512 ===
  OK   micro                got=0.8601 ssot=0.8601
  OK   macro                got=0.8572 ssot=0.8572
  OK   sample               got=0.8720 ssot=0.8720
  OK   empty_rate           got=0.0179 ssot=0.0179
  OK   anchor_weighted_f1   got=0.8203 ssot=0.8203

=== modernbert-patent-len8192 ===
  OK   micro                got=0.8685 ssot=0.8685
  OK   macro                got=0.8649 ssot=0.8649
  OK   sample               got=0.8825 ssot=0.8825
  OK   empty_rate           got=0.0135 ssot=0.0135
  OK   anchor_weighted_f1   got=0.8256 ssot=0.8256


In [11]:
for fp in sorted(Path(config["out_path"]).glob("*")):
    print(f"{fp.name:44s} {fp.stat().st_size / 1e6:8.1f} MB")

doc_ids_test.json                                 0.2 MB
doc_ids_val.json                                  0.2 MB
logits_kobert-patent-baseline_len512_test.npy      8.5 MB
logits_kobert-patent-baseline_len512_val.npy      8.4 MB
logits_modernbert-patent-len512_test.npy          8.5 MB
logits_modernbert-patent-len512_val.npy           8.4 MB
logits_modernbert-patent-len8192_test.npy         8.5 MB
logits_modernbert-patent-len8192_val.npy          8.4 MB
